In [ ]:
# Import usual Text Mining libraries 

import nltk
import numpy
import string
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.text import Text
import matplotlib.pyplot as plt
import pprint as pp
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('averaged_perceptron_tagger')
from nltk.tag import pos_tag
!pip install wordcloud
from wordcloud import WordCloud
import pandas as pd
!pip install tabulate
import matplotlib.lines as mlines
import matplotlib.ticker as ticker

In [ ]:
# Nltk is the natural language processing toolkit 

from nltk.corpus import PlaintextCorpusReader

# Point to the collection of documents (corpus)
corpus_root = "./data/inaugural"

# Reads the corpus into an object 
corpus_reader = PlaintextCorpusReader(corpus_root, '.*', encoding='latin1') 

print(corpus_reader)

In [ ]:
# Let's get the file ID so we can find the date and president name 

all_file_ids = corpus_reader.fileids()

# Check the file IDs 

print(all_file_ids[:5]) 
print(all_file_ids[-5:]) 

In [ ]:
# Let's remove the Read Me file as we don't need it 

# We use .strip() to remove any leading/trailing whitespace
# We use .lower() to ignore case (to catch 'README', 'readme', etc.)
# We check if the stripped, lowercased ID is 'readme'

new_file_ids = [
    file_id 
    for file_id in all_file_ids 
    if file_id.strip().lower() != 'readme'
]

# Show the length of the original fileid list 
print(f"Original list size: {len(all_file_ids)}")

# Check the length of the new file ids to see if we have removed the file 
print(f"New list size: {len(new_file_ids)}")

# Now, assign the new list back to your working variable (if needed)
file_ids = new_file_ids

# Display the last 5 file IDs (the file should be gone)
print(new_file_ids[-5:])


In [ ]:
type(new_file_ids)

In [ ]:
# So 'new_file_ids' holds the list of file IDs after removing 'readme'

# Let's make an empty container. This is to create a structure that will collect all the results.
# Python allocates a small space in its memory and we assign a name to this new, empty list, here corpus_data

corpus_data = []

# Now let's extract the speeches
# The code below controls the repetition. This entire block (the for statement) runs once for each file in my new_file_ds list. 
# We use the temporary variable corpus_data to facilitate that process 


for fileid in new_file_ids: 
   
# Here Python is using the corpus_reader object to locate the file by its fileid string.. 
# It reads the text content from our file, and stores the result in the raw_text temporary variable 
    
    raw_text = corpus_reader.raw(fileid)
    
# Now we want to structure and store the result 
# We Append a list containing the fileid AND raw_text to corpus_data
# .append() adds one item to the end of a list without overwriting the list itself 
# This keeps the speech text linked to its ID.
# Python creates a new list containing both the fileid (identifier) and the speech (raw_text)
# It adds this new two-list item to the end of the main corpus_data list
# Basically the list goes from '1789-Washington.txt' to [['1789-Washington.txt',raw_text_1]], which then repeats for each fileid 
# You see that here we're setting up the append to show the structure we want it to be in 
    
    corpus_data.append([fileid, raw_text])

# Verification
print(f"First speech's File ID: {corpus_data[0][0]}")
print(f"Start of first speech's text: {corpus_data[0][1][:100]}...")

In [ ]:
type(corpus_data)

In [ ]:
# Now it makes sense to make a dataframe

import pandas as pd

# Create the DataFrame using the list of lists
df = pd.DataFrame(corpus_data, columns=['File_ID', 'Raw_Speech_Text'])

# 2. Display the first few rows to verify the structure

print(df.head())
print('')
print(f"Total rows (speeches): {len(df)}")

In [ ]:
# Let's tidy up the column names 

# Let's duplicate and rename the original 'File_ID' column
df['Identifier'] = df['File_ID']

# Let's remove the .txt from the end of the identifier string 
df['Identifier'] = df['Identifier'].str.replace('.txt', '', regex=False)

# We split the 'Identifier' column by the hyphen ('-'). The .str.splt(-) method turns each string into a list of two elements wherever it finds a hypen. 
# The .str[0] and .str[1] then selects the two elements to create Year and President columns  
    

# .str[0] selects the first item (the Year)
df['Year'] = df['Identifier'].str.split('-').str[0]

# .str[1] selects the second item (the President's Name)
df['President'] = df['Identifier'].str.split('-').str[1]

# 4. Display the updated DataFrame to check the new columns
print("✅ New columns ('Identifier', 'Year', 'President') created:")
print(df[['File_ID', 'Identifier', 'Year', 'President']].head())



In [ ]:
print(df.head())

In [ ]:
# I felt a bit worried about dropping columns, as I wasn't sure it was going to preserve the speech with the right fileid
# But after doing some research I see that Python doesn't actually record which speeches are matched with which fileid, I just set it up that certain variables were on the same rows
# Ie. deleting them won't change anything, as it's not as if the 'fileid' was the master column or anything

In [ ]:
# So, let's drop the columns we don't need 

df = df.drop(columns=['Identifier'])

df = df.drop(columns=['File_ID']) 

# Let's also rename the Speech column
df = df.rename(columns={'Raw_Speech_Text': 'Speech'})

# Display the first few columns to verify the change
print("✅ Column successfully renamed.")
print(df[['Year', 'President', 'Speech']].head())

print("✅ Columns dropped. DataFrame structure updated.")
print(df.head())

In [ ]:
print(df.head())

In [ ]:
# Now I want to change the order of the columns 

# Create a list defining the order: Year first, then President, then the text

inauguralspeeches = [
    'Year',             
    'President',       
    'Speech'
]

In [ ]:
# This is actually a bit confusing to me. 
# Apparently doing this is a standard way of transforming the DataFrame whilst keeping the same variable names
# By default most operations in Pandas do not modify the original DF in place, it creates a new one
# So when I wrote df = df[inauguralspeeches] I was telling it to take the DF and overwrite the old DF. 

In [ ]:
df = df[inauguralspeeches]

In [ ]:
# Explanation: 
# The df[inauguralspeeches] means:  
    # hey python
    # take my inauguralspeeches list of strings ('Year' etc.)
    # look up the columns with those names within the existing dataframe - df 
    # make that into a dataframe with columns in the order specified by the list

# The original df isn't changed yet, just a copy is created 
# Any column not in the inauguralspeeches lis is excluded from the new dataframe 

# The df on the left side of the = means:
    # point our original dataframe 'df' to the new reordered dataframe

# The original dataframe object is discared from memory and my variable df is permanently updated to reflect the new column order and selection


In [ ]:
print(df.head())

In [ ]:
# Create a new column called 'Lower_Speech'

# We use .str.lower() to convert every string in the column to lowercase.
df['Lower_Speech'] = df['Speech'].str.lower()


In [ ]:
# Now let's move onto actually mining these speeches! 

In [ ]:
# First I need to tokenize the text so I can work with it
# Tokenizing means turning strings into a list of words and punctuation 

# Define the function explicitly
def simple_tokenize(text_string):

    # We first check if the value is missing (NaN)
    if pd.isna(text_string):
        return []
    
    # We return the direct output of word_tokenize
    return word_tokenize(text_string)

# Apply the function to the 'Speech' column that we made lowercase already 
df['Simple_Tokens'] = df['Lower_Speech'].apply(simple_tokenize)


In [ ]:
print(df[['Year', 'President', 'Simple_Tokens']].head())

In [ ]:
# The reason there's square brackets around the speech now is that the word_tokenize function converts a single string of text into a sequence of individual items (tokens)
# In Python a sequence of items is stored in a list, and lists always use square brackets 

In [ ]:
# Now let's focus on cleaning the speech data 

# import the stopwords package 

import string 
from nltk.corpus import stopwords

# Define the set of items to remove (this part stays the same)
remove_these = set(stopwords.words('english') + list(string.punctuation) + list(string.digits))

# Define the simpler function
def simple_clean_list(tokens):
    if not isinstance(tokens, list):
        return []
    
# The clean filtering logic:
    filtered_tokens = [
        token 
        for token in tokens 
        if token not in remove_these
    ]
    return filtered_tokens

# Apply the function to your 'Simple_Tokens' column
df['Cleaned_Tokens'] = df['Simple_Tokens'].apply(simple_clean_list)

print(df[['Year', 'President', 'Cleaned_Tokens']].head())

In [ ]:
# Define the function to tag and filter for superlatives
# Download tagger

nltk.download('averaged_perceptron_tagger')

# Tags tokens and filters them to return only superlatives

def extract_superlatives(tokens):
    if not isinstance(tokens, list) or not tokens:
        return []

    # Apply Part-of-Speech tagging
    tagged_tokens = nltk.pos_tag(tokens)

    # 2. Filter for adjectives (JJ, JJR, JJS tags)
    superlatives = [
        word
        for (word, pos) in tagged_tokens
        # We use .startswith('JJ') to include all adjective forms:
        # JJ (Adjective), JJR (Comparative), JJS (Superlative)
        if pos.startswith('JJS')
    ]
    return superlatives

# Apply the function to the 'Cleaned_Tokens' column and save the result
df['Superlatives'] = df['Cleaned_Tokens'].apply(extract_superlatives)

print(df[['Year', 'President', 'Superlatives']].head())

In [ ]:
print(df['Superlatives'])

In [ ]:
# Let's check these are actually all superlatives

all_superlatives = df['Superlatives'].sum()

all_superlatives

In [ ]:
# Removing words that follow the linguistic pattern of a superlative (-est) but aren't actually superlatives

to_remove = [ 'digest', 'forest', 'chest', 'west', 'test', 'manifest', 'lest', 'quest', 'harvest', 'rest', 'attest', 'tempest', 'wrest', 'suggest', 'earnest', 'honest', 'conquest', 'protest', 'budapest']

# Convert the list to a set for much faster lookups during filtering
remove_set = set(to_remove)

# Define the function to filter a single list of superlatives
def filter_false_superlatives(superlative_list):
    """Removes non-superlative words from a list of JJS-tagged tokens."""
    # Safety check for missing data
    if not isinstance(superlative_list, list) or not superlative_list:
        return []

    # Apply your filtering logic to the list of superlatives from one cell
    superlatives_clean = [
        token 
        for token in superlative_list
        if token not in remove_set
    ]
    return superlatives_clean

# Apply the function to the existing 'Superlatives' column and save the result
df['Superlatives_Clean'] = df['Superlatives'].apply(filter_false_superlatives)

print(df[['Year', 'President', 'Superlatives_Clean']].head())

In [ ]:
# Let's create a wordcloud
# All of the visualization tools (Counter, FreqDist, and WordCloud) are designed to work on flat data structures
# So one big list or one big string. 
# By using .sum(), we transform the DataFrame column (lists within cells) into the single, simple structure required.

from collections import Counter
from nltk.probability import FreqDist
from wordcloud import WordCloud 

# Combine all the lists in the 'Superlatives_Clean' column into one big list of words.
# This uses the .sum() method.

all_superlatives = df['Superlatives_Clean'].sum()

# Use the combined list ('all_superlatives') for Counter/FreqDist
fdist = FreqDist(all_superlatives)

# Plot the frequency distribution
fdist.plot(20, title='Frequency Distribution for the 20 Most Common Superlatives')
plt.show()


In [ ]:
# Let's also have a look at which superlatives are used the most 

from collections import Counter

# 1. Combine all the lists in the 'Superlatives_Clean' column into one big list
all_superlatives = df['Superlatives_Clean'].sum()

# 2. Count the frequency of each unique superlative
superlative_counts = Counter(all_superlatives)

# 3. Print the top 20 most common superlatives
print("--- Counts of All Superlatives in Inaugural Speeches ---")
print("\n")
for word, count in superlative_counts.most_common():
    print(f"'{word}': {count}")

In [ ]:
# Let's make a wordcloud
# Convert the single list of words into one single string for the WordCloud generator

text_for_wordcloud = ' '.join(all_superlatives)

# Generate and display the WordCloud
cloud = WordCloud(width=600, 
        height=400, 
        max_font_size=130, 
        colormap="bwr").generate(text_for_wordcloud)

plt.rcParams["figure.figsize"] = (16,12)
plt.imshow(cloud, interpolation='bilinear')
plt.axis('off')
plt.show()


In [ ]:
# Let's plot another graph, a bar chart this time, to show the total number of superlatives used by each President.
# The .apply(len) counts the items in the list (the superlatives) for each row.

df['Superlative_Count'] = df['Superlatives_Clean'].apply(len)

# Group the data by President and sum the counts.
superlative_counts_by_president = df.groupby('President')['Superlative_Count'].sum().sort_values(ascending=False)

# Create the bar chart.
plt.figure(figsize=(15, 5))
superlative_counts_by_president.plot(kind='bar', color='skyblue')
plt.title('Total Number of Superlatives Used by Each President')
plt.xlabel('President')
plt.ylabel('Total Superlative Count')
plt.xticks(rotation=45, ha='right') # Rotate names for better readability
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Ok I need to know the year of each speech as well 

# Ensure Superlative_Count column exists (if not already run)
df['Superlative_Count'] = df['Superlatives_Clean'].apply(len)

# Group by BOTH President and Year, and then sum the counts for that speech.
# This results in one row per speech.
superlative_counts_by_speech = df.groupby(['President', 'Year'])['Superlative_Count'].sum().reset_index()

# Sort the results for readability, perhaps by year.
superlative_counts_by_speech = superlative_counts_by_speech.sort_values(by='Superlative_Count', ascending=True)

print("✅ Data successfully grouped and sorted by President and Year.")
print(superlative_counts_by_speech)

In [ ]:
# Interesting! So now I can see that in my previous graph, it had said there were more superlatives because it grouped Presidents who made multiple speeches together. 

In [ ]:
# I think it could be good idea to add political parties in at this point. 

# Define the Party Mapping Dictionary
party_mapping = {
    # Early/Other (Federalist, Democratic-Republican, Whig)
    'Washington': 'Early/Other',
    'Adams': 'Early/Other',    # John Adams (Federalist) & John Quincy Adams (Dem-Republican)
    'Jefferson': 'Early/Other', # Democratic-Republican
    'Madison': 'Early/Other',  # Democratic-Republican
    'Monroe': 'Early/Other',   # Democratic-Republican
    'Jackson': 'Democrat',
    'VanBuren': 'Democrat',
    'Harrison': 'Early/Other',  # William Henry Harrison (Whig)
    'Polk': 'Democrat',
    'Taylor': 'Early/Other',   # Whig
    'Pierce': 'Democrat',
    'Buchanan': 'Democrat',
    
    # Modern Parties (Post-1860)
    'Lincoln': 'Republican',
    'Grant': 'Republican',
    'Hayes': 'Republican',
    'Garfield': 'Republican',
    'Cleveland': 'Democrat',
    'McKinley': 'Republican',
    'Roosevelt': 'Republican', # Theodore (1905)
    'Wilson': 'Democrat',
    'Harding': 'Republican',
    'Coolidge': 'Republican',
    'Hoover': 'Republican',
    
    # Note: FDR is 'Roosevelt' after 1933, so we'll use a complex mapping later if needed,
    # but for simplicity based on your list, we'll map by name.
    'Truman': 'Democrat',
    'Eisenhower': 'Republican',
    'Kennedy': 'Democrat',
    'Johnson': 'Democrat',
    'Nixon': 'Republican',
    'Carter': 'Democrat',
    'Reagan': 'Republican',
    'Bush': 'Republican',
    'Clinton': 'Democrat',
    'Obama': 'Democrat',
    'Trump': 'Republican',
    'Biden': 'Democrat',
}

# 2. Add the 'Party' column to the grouped DataFrame
superlative_counts_by_speech['Party'] = superlative_counts_by_speech['President'].map(party_mapping).fillna('Unknown')

# 3. Display the results with the new Party column
print("\n✅ Party column added. Results grouped by President, Year, and Party:")
print("---")
print(superlative_counts_by_speech.to_markdown(index=False))

In [ ]:
# It could be interesting to look at the superlative count per president, including their political party. 
# Lets replot that graph but including the political party and colour-code it. 


# --- 1. Data Preparation and Aggregation ---

# 1a. Define colors and mapping
party_colors = {
    'Democrat': '#0000FF',     # Blue
    'Republican': '#FF0000',   # Red
    'Early/Other': '#808080',  # Gray
    'Unknown': '#FFD700'       # Yellow/Gold for any missing Presidents 
}

party_mapping = {
    'Washington': 'Early/Other', 'Adams': 'Early/Other', 'Jefferson': 'Early/Other', 
    'Madison': 'Early/Other', 'Monroe': 'Early/Other', 'Jackson': 'Democrat', 
    'VanBuren': 'Democrat', 'Harrison': 'Early/Other', 'Polk': 'Democrat', 
    'Taylor': 'Early/Other', 'Pierce': 'Democrat', 'Buchanan': 'Democrat',
    'Lincoln': 'Republican', 'Grant': 'Republican', 'Hayes': 'Republican',
    'Garfield': 'Republican', 'Cleveland': 'Democrat', 'McKinley': 'Republican',
    'Roosevelt': 'Republican', 
    'Wilson': 'Democrat', 'Harding': 'Republican', 'Coolidge': 'Republican', 
    'Hoover': 'Republican', 'Truman': 'Democrat', 'Eisenhower': 'Republican',
    'Kennedy': 'Democrat', 'Johnson': 'Democrat', 'Nixon': 'Republican', 
    'Carter': 'Democrat', 'Reagan': 'Republican', 'Bush': 'Republican', 
    'Clinton': 'Democrat', 'Obama': 'Democrat', 'Trump': 'Republican', 
    'Biden': 'Democrat',
}

# 1b. Calculate total usage and add Party
df['Superlative_Count'] = df['Superlatives_Clean'].apply(len)
df['Party'] = df['President'].map(party_mapping).fillna('Unknown') 

# Group to get total count and the minimum year for sorting
df_total_usage = df.groupby(['President', 'Party']).agg(
    Total_Count=('Superlative_Count', 'sum'),
    Year=('Year', 'min') 
).reset_index()

# Sort the data by Year (ascending)
df_sorted = df_total_usage.sort_values(by='Year', ascending=True)

# --- CRITICAL CHANGE: Create combined labels (President + Year) ---
df_sorted['Label'] = df_sorted.apply(
    lambda row: f"{row['President']}\n({row['Year']})", 
    axis=1
)

# Create the final color list based on the chronological order
color_list = df_sorted['Party'].map(party_colors).tolist()


# --- 2. Matplotlib Plotting ---

plt.figure(figsize=(20, 8)) # Increase figure size slightly to accommodate two lines of text

# Use the new 'Label' column for the X-axis
plt.bar(df_sorted['Label'], df_sorted['Total_Count'], color=color_list)

plt.title('Total Superlative Usage by President (Chronological Order)', fontsize=16)
plt.xlabel('President (Inaugural Year)', fontsize=12)
plt.ylabel('Total Superlative Count', fontsize=12)

# Y-axis Integer Fix
plt.gca().yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Adjust rotation for the two-line labels—a small rotation or none is often better
plt.xticks(rotation=45, ha='right', fontsize=9) 
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Add a simple legend
legend_handles = [
    mlines.Line2D([], [], color=party_colors[party], marker='s', linestyle='None', markersize=10, label=party) 
    for party in party_colors
]
plt.legend(handles=legend_handles, title="Political Party", loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Let's look in more depth at which superlatives were used by which president. 

# Filter the DataFrame for the President 'Monroe'
monroe_speeches = df[df['President'] == 'Monroe']

# 2. Combine the list of superlatives from all his speeches (he likely only has one inaugural address)
monroe_superlatives = monroe_speeches['Superlatives_Clean'].sum()

# Print the results
print(f"President Monroes's total superlative count: {len(monroe_superlatives)}")
print("-" * 50)
print("Monroes's Superlatives (in order of appearance):")
print(van_buren_superlatives)

In [ ]:
 # Let's explore Martin Van Buren's speech more. 

# Filter the DataFrame for the President 'VanBuren'
van_buren_speeches = df[df['President'] == 'VanBuren']

# 2. Combine the list of superlatives from all his speeches (he likely only has one inaugural address)
van_buren_superlatives = van_buren_speeches['Superlatives_Clean'].sum()

# Print the results
print(f"President Van Buren's total superlative count: {len(van_buren_superlatives)}")
print("-" * 50)
print("Van Buren's Superlatives (in order of appearance):")
print(van_buren_superlatives)

In [ ]:
 # Let's explore Clinton's speech more. 

# Filter the DataFrame for the President 'VanBuren'
clinton_speeches = df[df['President'] == 'Clinton']

# 2. Combine the list of superlatives from all his speeches (he likely only has one inaugural address)
clinton_superlatives = clinton_speeches['Superlatives_Clean'].sum()

# Print the results
print(f"President Clinton's total superlative count: {len(clinton_superlatives)}")
print("-" * 50)
print("Clinton's Superlatives (in order of appearance):")
print(clinton_superlatives)

In [ ]:
 # Let's explore Coolidge's speech more. 

# Filter the DataFrame for the President 'VanBuren'
coolidge_speeches = df[df['President'] == 'Coolidge']

# 2. Combine the list of superlatives from all his speeches (he likely only has one inaugural address)
coolidge_superlatives = coolidge_speeches['Superlatives_Clean'].sum()

# Print the results
print(f"President Coolidges's total superlative count: {len(coolidge_superlatives)}")
print("-" * 50)
print("Coolidge's Superlatives (in order of appearance):")
print(coolidge_superlatives)

In [ ]:
from collections import Counter
import altair as alt

# Filter the DataFrame for the President 'VanBuren'
van_buren_superlatives_list = van_buren_speeches['Superlatives_Clean'].sum()

# Count the frequency of each unique word
superlative_counts = Counter(van_buren_superlatives_list)

# Convert the Counter object into a Pandas DataFrame for Altair
df_van_buren = pd.DataFrame(superlative_counts.items(), columns=['Superlative', 'Count'])

# Create the Bar Chart (sorted by count)
chart = alt.Chart(df_van_buren).mark_bar().encode(
    # X-axis: Superlative word (nominal type, sorted by Count descending)
    x=alt.X('Superlative:N', sort='-y', axis=alt.Axis(labelAngle=-45)),
    
    # Y-axis: Count (quantitative type)
    y=alt.Y('Count:Q'),
    
    # Tooltip to show data on hover
    tooltip=['Superlative', 'Count']
).properties(
    title="Superlative Frequency in President Van Buren's Inaugural Address"
)

In [ ]:
chart

In [ ]:
# What about Donald Trump's second speech?

# Filter the DataFrame for the President 'Trump' which is okay since he didn't use any superlatives in his first speech
trump_speech = df[df['President'] == 'Trump']

# 2. Combine the list of superlatives from all his speeches (he likely only has one inaugural address)
trump_superlatives = trump_speech['Superlatives_Clean'].sum()

# Print the results
print(f"President Trumps's total superlative count: {len(trump_superlatives)}")
print("-" * 50)
print("Trumps's Superlatives (in order of appearance):")
print(trump_superlatives)

# Filter the DataFrame for the President 'Trump'
trump_superlatives_list = trump_speech['Superlatives_Clean'].sum()

# Count the frequency of each unique word
trump_superlative_counts = Counter(trump_superlatives_list)

# Convert the Counter object into a Pandas DataFrame for Altair
df_trump = pd.DataFrame(trump_superlative_counts.items(), columns=['Superlative', 'Count'])

# Create the Bar Chart (sorted by count)
trump_chart = alt.Chart(df_trump).mark_bar().encode(
    # X-axis: Superlative word (nominal type, sorted by Count descending)
    x=alt.X('Superlative:N', sort='-y', axis=alt.Axis(labelAngle=-45)),
    
    # Y-axis: Count (quantitative type)
    y=alt.Y('Count:Q'),
    
    # Tooltip to show data on hover
    tooltip=['Superlative', 'Count']
).properties(
    title="Superlative Frequency in President Trump's 2025 Inaugural Address"
)

print("-" * 50)
trump_chart


In [ ]:
# Haha, so Trump really hammers it home in the 2025 address with 'greatest'

In [ ]:
# I might take a closer look at the concordances of the top 3 superlatives - namely 'best', 'greatest' and 'highest''

# Combine ALL cleaned tokens into one master list
full_corpus_tokens = df['Simple_Tokens'].sum()

# Wrap the list in the NLTK Text object
corpus_text_object = Text(full_corpus_tokens)

# Looking at 'best'
corpus_text_object.concordance("best")

In [ ]:


# Looking at 'greatest'
corpus_text_object.concordance("greatest")

In [ ]:
# Looking at 'highest'
corpus_text_object.concordance("highest")

In [ ]:
# It's quite interesting to examine the need for a 'the' infront of a superlative for emphasis. What even is that called linguistically? 
# I'm going to look at the other concordances within the superlatives and see what they say. 

In [ ]:
# Looking at 'least'
corpus_text_object.concordance("least")

In [ ]:
# Interesting, so here it's actually more the phrase 'at least' rather than 'the least'

In [ ]:
# Looking at 'surest'
corpus_text_object.concordance("surest")

# This is a good 'the' example

In [ ]:
# Looking at 'strongest'
corpus_text_object.concordance("strongest")

# Also good example 

In [ ]:
# Looking at 'largest'
corpus_text_object.concordance("largest")

# So this one does use the 'the' but it doesn't seem so impactful in terms of what it's referring to. Largest is more a quantifier whereas the other superlatives seem like indicators of strength? 

In [ ]:
# Looking at 'slightest'
corpus_text_object.concordance("slightest")

In [ ]:
# Looking at 'wisest'
corpus_text_object.concordance("wisest")

In [ ]:
# Looking at 'earliest'
corpus_text_object.concordance("earliest")

In [ ]:
# It's interesting to think about the use of 'the' before a superlative, and why this is necessary linguistically. 

In [ ]:
# From looking at the concordancdes I think it'd be most interesting to focus on 'the greatest'. It's also quite politically relevant to MAGA. 
# First let's check the counts of which presidents actually use the saying 'the greatest'

def count_the_greatest(token_list):
    count = 0
    
    # Iterate through the list of tokens that still contains 'the' (e.g., 'Simple_Tokens')
    for i in range(1, len(token_list)):
        current_word = token_list[i].lower()
        previous_word = token_list[i-1].lower()
        
        # Check for the specific sequence
        if previous_word == 'the' and current_word == 'greatest':
            count += 1
            
    return count

# Apply the function to create your final data column
df['The_Greatest_Count'] = df['Simple_Tokens'].apply(count_the_greatest)

# Display your results to find your comparison points
print(df[['President', 'Year', 'The_Greatest_Count']].sort_values(by='The_Greatest_Count', ascending=False))

In [ ]:
# Now let's extract the phrases themselves 

def extract_the_greatest_phrases(token_list):
    phrases = []
    
    # We must iterate up to len(token_list) - 2 because we need to look two words ahead
    for i in range(len(token_list) - 2):
        prev_word = token_list[i].lower()
        curr_word = token_list[i+1].lower()
        next_word = token_list[i+2].lower()

        # Check for the sequence: 'the' immediately followed by 'greatest'
        if prev_word == 'the' and curr_word == 'greatest':
            # Store the full phrase as a string
            phrases.append(f"the greatest {next_word}")

    return phrases

# Apply the function to create the new column listing all phrases per speech
df['The_Greatest_Phrases'] = df['Simple_Tokens'].apply(extract_the_greatest_phrases)

# Display the summary of speeches that contain the phrase
df_phrases_summary = df[df['The_Greatest_Phrases'].str.len() > 0][['President', 'Year', 'The_Greatest_Phrases']]

In [ ]:
df['The_Greatest_Phrases'] = df['Simple_Tokens'].apply(extract_the_greatest_phrases)

In [ ]:
# Create a DataFrame containing ONLY the speeches that used the phrase at least once
# We filter for rows where the list in 'The_Greatest_Phrases' has a length > 0

all_speeches_with_phrase = df[df['The_Greatest_Phrases'].str.len() > 0]

# Print all columns relevant to the analysis for all found speeches
print("--- All Speeches Containing 'The Greatest...' Phrase ---")
print(all_speeches_with_phrase[['President', 'Year', 'The_Greatest_Count', 'The_Greatest_Phrases']])

# To check the total number of speeches that contained the phrase:
print(f"\nTotal speeches containing the phrase: {len(all_speeches_with_phrase)}")

In [ ]:
print(all_speeches_with_phrase[['President', 'The_Greatest_Phrases']])

In [ ]:
# 1. Explode the list column to create one row for every single phrase instance
df_exploded = df.explode('The_Greatest_Phrases')

# 2. Filter out rows where the phrase was not found (NaN or empty strings if your data is slightly different)
df_claims = df_exploded.dropna(subset=['The_Greatest_Phrases'])

# 3. Count the frequency of each unique phrase across all speeches
claim_counts = Counter(df_claims['The_Greatest_Phrases'])
df_claim_counts = pd.DataFrame(claim_counts.most_common(), columns=['Claim', 'Count'])

# Group the exploded data by both President and Claim, and count the occurrences
president_claim_breakdown = df_claims.groupby(['Year','President', 'The_Greatest_Phrases', 'Party']).size().reset_index(name='Claim_Count')

print(president_claim_breakdown)


In [ ]:
# Assuming 'president_claim_breakdown' is available from your previous code block.

# 1. Aggregate the breakdown to get the TOTAL count per President/Party
df_total_usage = president_claim_breakdown.groupby(
    ['President', 'Party']
)['Claim_Count'].sum().reset_index(name='Total_Claims')

# 2. Define standard political colors for clarity
color_scale = alt.Scale(
    domain=['Democrat', 'Republican', 'Early/Other'],
    range=['#0000FF', '#FF0000', '#808080'] # Blue, Red, Gray
)

# 3. Create the Bar Chart
greatest_chart = alt.Chart(df_total_usage).mark_bar().encode(
    # X-axis: President (sorted by the total count for easy comparison)
    x=alt.X('President:N', sort='-y', title="President"),
    
    # Y-axis: Total Usage Count
    y=alt.Y('Total_Claims:Q', title='Count'),
    
    # Color: Political Party
    color=alt.Color('Party:N', scale=color_scale, legend=alt.Legend(title="Party")),
    
    # Tooltips: Show key info on hover
    tooltip=['President:N', 'Party:N', 'Total_Claims:Q']
).properties(
    title='"The Greatest..." claim within inaugural speech'
).interactive() # Allows zooming and panning

greatest_chart


In [ ]:
# Now I want to show the complete sentence containing 'the greatest' from each of these speeches. 

# Create a DataFrame containing ONLY the speeches that used the phrase at least once
df_speeches_with_greatest = df[df['The_Greatest_Count'] > 0].copy()

print("--- Full Sentences Containing 'The Greatest' ---")

# Iterate through the rows of the filtered DataFrame
for index, row in df_speeches_with_greatest.iterrows():
    president = row['President']
    year = row['Year']
    token_list = row['Simple_Tokens']  # Use the token list with 'the'

    # The NLTK Text object must be created for each speech
    speech_text = Text(token_list)

    print(f"\nPRESIDENT: {president} ({year})")
    print("-" * 30)

    # The concordance function prints the results directly
    # The width argument controls the amount of surrounding text shown
    speech_text.concordance("greatest", width=120)